# Decision Tree Classifier Example (Wine Quality Dataset)

**Goal: Classify wine into three quality tiers (Low 3-4, Mid 5-6, High 7-8) based on physicochemical features.**

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

# NOTEBOOK_DIR resolves to the folder this notebook lives in.
# Jupyter sets the working directory to wherever it was launched from,
# so we use __file__ would not work — os.path.abspath('') is the correct
# approach for Jupyter notebooks.
# Find the repo root reliably on any machine.
# We search upward from the current working directory until we find
# the 'data' folder, which only exists at the repo root.
def _find_repo_root():
    path = os.path.abspath(os.getcwd())
    for _ in range(10):  # search up to 10 levels up
        if os.path.isdir(os.path.join(path, 'data')) and os.path.isdir(os.path.join(path, 'src')):
            return path
        path = os.path.dirname(path)
    raise RuntimeError(
        "Could not find repo root. Make sure you launched Jupyter from inside the CMOR-438 folder."
    )

REPO_ROOT = _find_repo_root()

# Algorithm source files live in src/supervised/ at the repo root,
# which is four levels up from examples/supervised/<algo>/
SRC_SUP  = os.path.join(REPO_ROOT, 'src', 'supervised')
sys.path.insert(0, SRC_SUP)

# Data files live in data/ at the repo root
DATA_DIR = os.path.join(REPO_ROOT, 'data')
from decision_trees import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

wine = pd.read_csv(os.path.join(DATA_DIR, 'WineQT.csv')).drop(columns=['Id'])
FEATURE_COLS = [c for c in wine.columns if c != 'quality']
print(f"Dataset loaded: {wine.shape[0]} samples, {len(FEATURE_COLS)} features.")

## 2. Preprocessing

In [ ]:
X = StandardScaler().fit_transform(wine[FEATURE_COLS].values.astype(float))
y_clf = np.array([0 if q<=4 else (1 if q<=6 else 2) for q in wine['quality'].values])
X_tr, X_te, y_tr, y_te = train_test_split(X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
print(f"Training samples: {X_tr.shape[0]}  |  Test samples: {X_te.shape[0]}")
print(f"Class distribution: {dict(zip(*np.unique(y_clf, return_counts=True)))}") 

## 3. Train

In [ ]:
dtc = DecisionTreeClassifier(criterion='gini', max_depth=8, min_samples_leaf=4)
dtc.fit(X_tr, y_tr)
print(f'Accuracy: {dtc.accuracy(X_te, y_te):.4f}')
print(f'Actual tree depth: {dtc.get_depth()}')

## 4. Results and Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

idx = np.argsort(dtc.feature_importances_)[::-1]
colors = plt.cm.viridis(np.linspace(0.2, 0.85, len(FEATURE_COLS)))
axes[0].bar(range(len(FEATURE_COLS)), dtc.feature_importances_[idx], color=colors, edgecolor='white')
axes[0].set_xticks(range(len(FEATURE_COLS)))
axes[0].set_xticklabels([FEATURE_COLS[i] for i in idx], rotation=40, ha='right', fontsize=8)
axes[0].set_ylabel('Gini Importance')
axes[0].set_title('Decision Tree - Feature Importances', fontweight='bold')

depths = range(2, 16)
depth_accs = []
for d in depths:
    m = DecisionTreeClassifier(criterion='gini', max_depth=d, min_samples_leaf=4)
    m.fit(X_tr, y_tr)
    depth_accs.append(m.accuracy(X_te, y_te))
axes[1].plot(list(depths), depth_accs, 's-', color='seagreen', lw=1.5, ms=6)
axes[1].axvline(8, color='red', linestyle='--', lw=1.2, label='Chosen depth=8')
axes[1].set_xlabel('Max Depth'); axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Decision Tree - Depth vs Accuracy', fontweight='bold')
axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Analysis

**Result: 84.7% test accuracy** on the 3-class quality tier task at depth 8.

**Feature importances** reveal which physicochemical properties most strongly separate the quality tiers. Alcohol content typically ranks first or second — high-quality wines tend to have higher alcohol. Volatile acidity (which gives wine an unpleasant vinegar taste at high levels) is usually the second most important feature. Sulphates, which act as a preservative and contribute to wine body, also rank highly. Free and total sulfur dioxide, residual sugar, and density typically rank lower.

**The depth vs accuracy sweep** is the most instructive plot. At depth 2–3 the tree is too shallow to capture meaningful interactions (underfitting). Accuracy rises sharply to around depth 7–8, then plateaus or even declines slightly as the tree begins to overfit the training data. The chosen depth of 8 sits at or near this peak.

**Class imbalance challenge:** with only 39 Low-quality wines (3.4% of the dataset), the tree will rarely see examples of Class 0 during training. This means Low-quality wines are likely the most common misclassification — the tree defaults to predicting Mid or High for borderline cases.

**Actual depth of 8** matching the max_depth setting confirms the tree used its full allowance, meaning the min_samples_leaf=4 constraint (not depth) is the binding regularisation at some nodes.

**Key takeaway:** The decision tree is interpretable — you can trace any prediction through a sequence of if/else rules — but it is a single model and therefore more prone to overfitting than ensembles. The feature importance ranking is valuable for understanding which wine properties actually drive perceived quality.